In [1]:
import os, random, time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
AUG_ROOT = "D:/mamba_model/aug_clean_tio"       
TAG = "tio_crossattn"                             
# ───────────────────────────────────────────────────────────────

COHORT_CSV = "D:/mamba_model/thesis_cohort_clean.csv"
MRI_CACHE  = f"{AUG_ROOT}/roi_mri"
PET_CACHE  = f"{AUG_ROOT}/roi_pet"
CKPT_DIR   = f"D:/mamba_model/checkpoints_v7_roi_{TAG}"
RESULTS    = f"D:/mamba_model/v7_roi_{TAG}_results.json"
os.makedirs(CKPT_DIR, exist_ok=True)

SPLIT_SEED  = 42
AUG_SEEDS   = [1, 101, 42]
BATCH_SIZE  = 1
NUM_WORKERS = 0          

print(f"aug:  {AUG_ROOT}")
print(f"ckpt: {CKPT_DIR}")
for p in (MRI_CACHE, PET_CACHE):
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f"  {os.path.basename(p)}: {n} files{'  *** MISSING ***' if n == 0 else ''}")

aug:  D:/mamba_model/aug_clean_tio
ckpt: D:/mamba_model/checkpoints_v7_roi_tio_crossattn
  roi_mri: 560 files
  roi_pet: 560 files


In [3]:
class VimEncoder(nn.Module):
    """Bidirectional Mamba over a token sequence. No CNN, no pretraining."""
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        cfg = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                          bidirectional=True, divide_output=True,
                          pscan=True, use_cuda=False)
        self.encoder = VMamba(cfg)
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [4]:
class ROIPatchEmbed3D(nn.Module):
    """6 ROIs -> non-overlapping 8^3 patches -> one token each. 3072 tokens."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32):
        super().__init__()
        self.n_rois, self.patch_size = n_rois, patch_size
        self.grid_size = roi_size // patch_size
        self.patches_per_roi = self.grid_size ** 3
        self.d_model = d_model
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.roi_embed    = nn.Embedding(n_rois, d_model)
        self.depth_embed  = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed  = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            for e in [self.roi_embed, self.depth_embed, self.height_embed, self.width_embed]:
                e.weight.mul_(0.02)
        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                 torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], -1).reshape(-1, 3),
                             persistent=False)

    def forward(self, rois):
        B, n = rois.shape[:2]
        x = rois.reshape(B * n, 1, *rois.shape[-3:])
        tokens = self.patch_conv(x).flatten(2).transpose(1, 2)
        tokens = tokens.reshape(B, n, self.patches_per_roi, self.d_model)
        c = self.coordinates
        spatial = (self.depth_embed(c[:, 0]) + self.height_embed(c[:, 1])
                   + self.width_embed(c[:, 2]))
        tokens = tokens + spatial[None, None] + self.roi_embed.weight[None, :, None, :]
        occ = F.max_pool3d((x.abs() > 1e-6).float(),
                           kernel_size=self.patch_size, stride=self.patch_size)
        valid = occ.flatten(1).bool().reshape(B, n, self.patches_per_roi)
        tokens = tokens.reshape(B, -1, self.d_model)
        valid = valid.reshape(B, -1)
        return tokens * valid.unsqueeze(-1).to(tokens.dtype), valid


class CrossModalVisionMambaModel(nn.Module):
    """MRI and PET tokens attend to each other before pooling.."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.n_rois = n_rois
        self.patches_per_roi = (roi_size // patch_size) ** 3

        self.mri_patch_embed = ROIPatchEmbed3D(n_rois, roi_size, patch_size, d_model)
        self.pet_patch_embed = ROIPatchEmbed3D(n_rois, roi_size, patch_size, d_model)
        self.mri_vim = VimEncoder(d_model, n_layers, d_state)
        self.pet_vim = VimEncoder(d_model, n_layers, d_state)

        self.cross_attn_mri_to_pet = nn.MultiheadAttention(d_model, num_heads=4,
                                                           dropout=dropout,
                                                           batch_first=True)
        self.cross_attn_pet_to_mri = nn.MultiheadAttention(d_model, num_heads=4,
                                                           dropout=dropout,
                                                           batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)

    def forward(self, mri_rois, pet_rois, return_attention=False):
        mri_tokens, mri_valid = self.mri_patch_embed(mri_rois)
        pet_tokens, pet_valid = self.pet_patch_embed(pet_rois)

        mri_tokens = self.dropout(self.mri_vim(mri_tokens))
        pet_tokens = self.dropout(self.pet_vim(pet_tokens))

        mri_attended, pet_to_mri_w = self.cross_attn_pet_to_mri(
            pet_tokens, mri_tokens, mri_tokens, key_padding_mask=~mri_valid,
            need_weights=return_attention, average_attn_weights=True)
        pet_attended, mri_to_pet_w = self.cross_attn_mri_to_pet(
            mri_tokens, pet_tokens, pet_tokens, key_padding_mask=~pet_valid,
            need_weights=return_attention, average_attn_weights=True)

        wm = mri_valid.unsqueeze(-1).to(mri_attended.dtype)
        wp = pet_valid.unsqueeze(-1).to(pet_attended.dtype)
        mri_pooled = (mri_attended * wp).sum(1) / wp.sum(1).clamp_min(1.0)
        pet_pooled = (pet_attended * wm).sum(1) / wm.sum(1).clamp_min(1.0)

        logits = self.classifier(self.dropout(torch.cat([mri_pooled, pet_pooled], dim=1)))
        if return_attention:
            return logits, pet_to_mri_w, mri_to_pet_w
        return logits


_m = CrossModalVisionMambaModel()
print(f"params: {sum(p.numel() for p in _m.parameters()):,}")
del _m

params: 98,370


In [5]:
df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values

X_tv, X_test, y_tv, y_test = train_test_split(
    sessions, labels, test_size=0.2, random_state=SPLIT_SEED, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=SPLIT_SEED, stratify=y_tv)

session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))
print(f"train {len(X_train)} | val {len(X_val)} | test {len(X_test)} "
      f"| test pos {int(y_test.sum())}")


# ── in-memory cache: first epoch reads from disk, the rest from RAM ──
_CACHE = {}

def load_cached(path):
    a = _CACHE.get(path)
    if a is None:
        a = np.load(path).astype(np.float32)
        _CACHE[path] = a
    return a
# NOTE: torch.from_numpy shares memory with the cached array.


class ROIDataset(Dataset):
    """Single modality. Kept for compatibility -- cross-modal attention
    requires both modalities, so only mm_loaders is used in this notebook."""
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples, self.cache_dir = [], cache_dir
        for ses, lab in zip(sessions, labels):
            key = ses if is_mri else session_to_subject[ses]
            self.samples.append((key, lab, "orig"))
            if is_train:
                for s in AUG_SEEDS:
                    self.samples.append((key, lab, f"aug{s}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        key, lab, ver = self.samples[i]
        a = load_cached(f"{self.cache_dir}/{key}_{ver}.npy")
        return torch.from_numpy(a).unsqueeze(1), torch.tensor(lab, dtype=torch.long), key


class MultimodalROIDataset(Dataset):
    """Pairs MRI and PET for the same subject and the same augmentation seed."""
    def __init__(self, sessions, labels, mri_dir, pet_dir, is_train=False):
        self.samples, self.mri_dir, self.pet_dir = [], mri_dir, pet_dir
        for ses, lab in zip(sessions, labels):
            sid = session_to_subject[ses]
            self.samples.append((ses, sid, lab, "orig"))
            if is_train:
                for s in AUG_SEEDS:
                    self.samples.append((ses, sid, lab, f"aug{s}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        mk, pk, lab, ver = self.samples[i]
        m = load_cached(f"{self.mri_dir}/{mk}_{ver}.npy")
        p = load_cached(f"{self.pet_dir}/{pk}_{ver}.npy")
        return (torch.from_numpy(m).unsqueeze(1), torch.from_numpy(p).unsqueeze(1),
                torch.tensor(lab, dtype=torch.long), mk)


def dl(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=(NUM_WORKERS > 0))

mri_loaders = (dl(ROIDataset(X_train, y_train, MRI_CACHE, True, True), True),
               dl(ROIDataset(X_val,   y_val,   MRI_CACHE, True, False)),
               dl(ROIDataset(X_test,  y_test,  MRI_CACHE, True, False)))

pet_loaders = (dl(ROIDataset(X_train, y_train, PET_CACHE, False, True), True),
               dl(ROIDataset(X_val,   y_val,   PET_CACHE, False, False)),
               dl(ROIDataset(X_test,  y_test,  PET_CACHE, False, False)))

mm_loaders  = (dl(MultimodalROIDataset(X_train, y_train, MRI_CACHE, PET_CACHE, True), True),
               dl(MultimodalROIDataset(X_val,   y_val,   MRI_CACHE, PET_CACHE, False)),
               dl(MultimodalROIDataset(X_test,  y_test,  MRI_CACHE, PET_CACHE, False)))

print(f"train samples (4x augmented): {len(mm_loaders[0].dataset)}")

t0 = time.time()
for i, _ in enumerate(mm_loaders[0]):
    if i >= 20: break
cold = time.time() - t0
t0 = time.time()
for i, _ in enumerate(mm_loaders[0]):
    if i >= 20: break
warm = time.time() - t0
print(f"20 batches: cold {cold:.1f}s -> warm {warm:.1f}s")
print(f"cached arrays: {len(_CACHE)} (~{sum(a.nbytes for a in _CACHE.values())/1e9:.1f} GB)")

train 120 | val 40 | test 40 | test pos 20
train samples (4x augmented): 480
20 batches: cold 2.1s -> warm 1.5s
cached arrays: 74 (~0.5 GB)


In [6]:
def train_epoch(model, loader, opt, crit, mm):
    model.train(); tot = 0
    for batch in loader:
        opt.zero_grad()
        if mm:
            a, b, lb, _ = batch; out = model(a.to(device), b.to(device))
        else:
            a, lb, _ = batch;    out = model(a.to(device))
        loss = crit(out, lb.to(device)); loss.backward(); opt.step(); tot += loss.item()
    return tot / len(loader)


def evaluate(model, loader, crit, mm):
    model.eval(); tot, P, L = 0, [], []
    with torch.no_grad():
        for batch in loader:
            if mm:
                a, b, lb, _ = batch; out = model(a.to(device), b.to(device))
            else:
                a, lb, _ = batch;    out = model(a.to(device))
            tot += crit(out, lb.to(device)).item()
            P.extend(out.argmax(1).cpu().numpy()); L.extend(lb.numpy())
    return (tot/len(loader), np.mean(np.array(P) == np.array(L)),
            recall_score(L, P, zero_division=0), specificity_score(L, P))


def measure_inference(model, loader, mm, n=20):
    model.eval(); ts = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n: break
            if mm:
                a, b = batch[0].to(device), batch[1].to(device); bs = a.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time(); _ = model(a, b)
            else:
                a = batch[0].to(device); bs = a.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time(); _ = model(a)
            if device.type == 'cuda': torch.cuda.synchronize()
            ts.append((time.time() - t0) / bs)
    return np.mean(ts), np.std(ts)


def compute_flops(model, loader, mm):
    try:
        model.eval(); b = next(iter(loader))
        with torch.no_grad():
            inp = (b[0][:1].to(device), b[1][:1].to(device)) if mm else (b[0][:1].to(device),)
            macs, _ = profile(model, inputs=inp, verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs failed: {e})"); return None


def run_seed(seed, model_cls, loaders, mm, prefix,
             max_epochs=101, patience=15, min_epochs=25, lr=1e-4, log_every=5):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    np.random.seed(seed); random.seed(seed)
    tr, va, te = loaders

    model = model_cls(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    crit  = nn.CrossEntropyLoss(label_smoothing=0.05)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sch   = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)

    best, no_imp, best_ep, total = float('inf'), 0, 0, 0
    path = f"{CKPT_DIR}/{prefix}_seed{seed}.pt"
    print(f"\n--- {prefix} seed {seed} ---")

    for ep in range(1, max_epochs):
        t0 = time.time()
        trl = train_epoch(model, tr, opt, crit, mm)
        vl, vacc, vtpr, vtnr = evaluate(model, va, crit, mm)
        sch.step(vl); dt = time.time() - t0; total += dt
        if ep % log_every == 0 or ep == 1:
            print(f"  ep {ep:>3} | train {trl:.4f} | val {vl:.4f} | "
                  f"acc {vacc:.3f} tpr {vtpr:.3f} tnr {vtnr:.3f} | {dt:.0f}s")
        if vl < best:
            best, best_ep, no_imp = vl, ep, 0
            torch.save(model.state_dict(), path)
        else:
            no_imp += 1
            if ep >= min_epochs and no_imp >= patience:
                print(f"  early stop {ep}, best {best_ep}"); break

    if best_ep < 5:
        print(f"  WARNING: best epoch {best_ep} -- may not have trained")

    model.load_state_dict(torch.load(path, weights_only=True))
    _, acc, tpr, tnr = evaluate(model, te, crit, mm)
    npar = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_m, inf_s = measure_inference(model, te, mm)
    fl = compute_flops(model, te, mm)

    print(f"  >>> TEST Acc={acc*100:.1f}% TPR={tpr*100:.1f}% TNR={tnr*100:.1f}% | "
          f"params={npar:,} train={total/60:.1f}min inf={inf_m*1000:.2f}ms "
          f"{f'{fl/1e9:.2f}GFLOPs' if fl else ''} best_ep={best_ep}")

    return {"seed": seed, "acc": acc, "tpr": tpr, "tnr": tnr, "best_epoch": best_ep,
            "train_time_sec": total, "n_params": npar,
            "inf_time_ms": inf_m*1000, "inf_std_ms": inf_s*1000, "flops": fl}


results = {"mri": [], "pet": [], "mm": []}
print("ready")

ready


In [7]:
results["mm"].append(run_seed(1, CrossModalVisionMambaModel, mm_loaders, True, "v7_roi_crossattn_mm"))


--- v7_roi_crossattn_mm seed 1 ---
  ep   1 | train 0.6967 | val 0.6831 | acc 0.550 tpr 0.850 tnr 0.250 | 116s
  ep   5 | train 0.6690 | val 0.6449 | acc 0.700 tpr 0.450 tnr 0.950 | 70s
  ep  10 | train 0.5731 | val 0.5624 | acc 0.675 tpr 0.750 tnr 0.600 | 67s
  ep  15 | train 0.3521 | val 0.5404 | acc 0.675 tpr 0.800 tnr 0.550 | 67s
  ep  20 | train 0.1826 | val 0.7503 | acc 0.750 tpr 0.500 tnr 1.000 | 68s
  ep  25 | train 0.1502 | val 0.6541 | acc 0.800 tpr 0.650 tnr 0.950 | 69s
  ep  30 | train 0.1337 | val 0.5998 | acc 0.725 tpr 0.700 tnr 0.750 | 67s
  early stop 31, best 16
  >>> TEST Acc=62.5% TPR=60.0% TNR=65.0% | params=98,370 train=35.6min inf=39.92ms 0.47GFLOPs best_ep=16


In [8]:
results["mm"].append(run_seed(7, CrossModalVisionMambaModel, mm_loaders, True, "v7_roi_crossattn_mm"))


--- v7_roi_crossattn_mm seed 7 ---
  ep   1 | train 0.6982 | val 0.6895 | acc 0.500 tpr 1.000 tnr 0.000 | 69s
  ep   5 | train 0.6760 | val 0.6574 | acc 0.700 tpr 0.450 tnr 0.950 | 64s
  ep  10 | train 0.6032 | val 0.5785 | acc 0.750 tpr 0.700 tnr 0.800 | 67s
  ep  15 | train 0.4930 | val 0.5204 | acc 0.725 tpr 0.850 tnr 0.600 | 69s
  ep  20 | train 0.3298 | val 0.5741 | acc 0.725 tpr 0.900 tnr 0.550 | 65s
  ep  25 | train 0.2116 | val 0.8976 | acc 0.625 tpr 0.950 tnr 0.300 | 67s
  ep  30 | train 0.1810 | val 0.5375 | acc 0.725 tpr 0.850 tnr 0.600 | 67s
  ep  35 | train 0.1588 | val 0.5398 | acc 0.700 tpr 0.800 tnr 0.600 | 30s
  ep  40 | train 0.1446 | val 0.5342 | acc 0.725 tpr 0.750 tnr 0.700 | 28s
  early stop 42, best 27
  >>> TEST Acc=67.5% TPR=75.0% TNR=60.0% | params=98,370 train=41.7min inf=18.62ms 0.47GFLOPs best_ep=27


In [9]:
results["mm"].append(run_seed(123, CrossModalVisionMambaModel, mm_loaders, True, "v7_roi_crossattn_mm"))


--- v7_roi_crossattn_mm seed 123 ---
  ep   1 | train 0.6960 | val 0.6848 | acc 0.700 tpr 0.450 tnr 0.950 | 28s
  ep   5 | train 0.6660 | val 0.6470 | acc 0.675 tpr 0.600 tnr 0.750 | 27s
  ep  10 | train 0.6061 | val 0.6048 | acc 0.700 tpr 0.450 tnr 0.950 | 28s
  ep  15 | train 0.4452 | val 0.5046 | acc 0.750 tpr 0.750 tnr 0.750 | 29s
  ep  20 | train 0.2611 | val 0.5527 | acc 0.750 tpr 0.750 tnr 0.750 | 30s
  ep  25 | train 0.1402 | val 0.6475 | acc 0.750 tpr 0.700 tnr 0.800 | 27s
  ep  30 | train 0.1298 | val 0.6169 | acc 0.750 tpr 0.750 tnr 0.750 | 29s
  early stop 30, best 15
  >>> TEST Acc=62.5% TPR=60.0% TNR=65.0% | params=98,370 train=14.2min inf=18.88ms 0.47GFLOPs best_ep=15


In [10]:
INCLUDE_SEEDS = [1, 7, 123]

def summarize(rs, name, include=INCLUDE_SEEDS):
    rs = [r for r in rs if r['seed'] in include]
    if not rs: print(f"{name}: no runs"); return
    a = [r['acc'] for r in rs]; t = [r['tpr'] for r in rs]; n = [r['tnr'] for r in rs]
    tm = [r['train_time_sec'] for r in rs]; inf = [r['inf_time_ms'] for r in rs]
    sd = (lambda v: np.std(v, ddof=1)*100 if len(v) > 1 else 0.0)
    print(f"{name}: Acc={np.mean(a)*100:.1f}±{sd(a):.1f}% "
          f"(range {min(a)*100:.1f}-{max(a)*100:.1f}) | "
          f"TPR={np.mean(t)*100:.1f}±{sd(t):.1f}% | TNR={np.mean(n)*100:.1f}±{sd(n):.1f}% | "
          f"Params={rs[0]['n_params']:,} | Train={np.mean(tm)/60:.1f}m | "
          f"Inf={np.mean(inf):.2f}ms | seeds={[r['seed'] for r in rs]}")
    print("    per seed: " + ", ".join(
        f"seed {r['seed']} {r['acc']*100:.1f}% (TPR {r['tpr']*100:.1f} / "
        f"TNR {r['tnr']*100:.1f}, best_ep {r['best_epoch']})" for r in rs))

print(f"=== v7 EARLY cross-modal attention — {TAG}, 200-subject cohort ===")
print("    all 3,072 tokens per modality attend across modalities before pooling")
summarize(results['mm'], 'Multimodal')


with open(RESULTS, 'w') as f:
    json.dump({k: [{kk: (float(vv) if isinstance(vv, (float, np.floating)) else vv)
                    for kk, vv in r.items()} for r in v] for k, v in results.items()},
              f, indent=2)
print(f"\nsaved {RESULTS}")

=== v7 EARLY cross-modal attention — tio_crossattn, 200-subject cohort ===
    all 3,072 tokens per modality attend across modalities before pooling
Multimodal: Acc=64.2±2.9% (range 62.5-67.5) | TPR=65.0±8.7% | TNR=63.3±2.9% | Params=98,370 | Train=30.5m | Inf=25.81ms | seeds=[1, 7, 123]
    per seed: seed 1 62.5% (TPR 60.0 / TNR 65.0, best_ep 16), seed 7 67.5% (TPR 75.0 / TNR 60.0, best_ep 27), seed 123 62.5% (TPR 60.0 / TNR 65.0, best_ep 15)

saved D:/mamba_model/v7_roi_tio_crossattn_results.json


In [11]:
#  GFLOPs for one forward pass, batch size 1 -- early cross-modal attention

#  Architecture alone determines these figures, so no training, checkpoints
#  or data are needed.

import copy
from torch.utils.flop_counter import FlopCounterMode
from mambapy.vim import VMamba as _VMamba


class _AttnStub(nn.Module):
    def __init__(self, log, d, batch_first):
        super().__init__(); self.log, self.d, self.bf = log, d, batch_first
    def forward(self, q, k, v, **kw):
        Lq = q.shape[1] if self.bf else q.shape[0]
        Lk = k.shape[1] if self.bf else k.shape[0]
        self.log.append((Lq, Lk, self.d))
        return torch.zeros_like(q), None


def count_flops(model, inputs):
    m = copy.deepcopy(model).eval().cpu()
    inputs = [t.detach().cpu() for t in inputs]
    attn_log, mamba_log, handles = [], [], []

    def mk(mod):
        def hook(_, inp, __):
            c = mod.config
            mamba_log.append(dict(L=inp[0].shape[1], ed=c.d_inner, n=c.d_state,
                                  layers=c.n_layers,
                                  bi=getattr(c, "bidirectional", False)))
        return hook
    for mod in m.modules():
        if isinstance(mod, _VMamba):
            handles.append(mod.register_forward_hook(mk(mod)))

    def swap(parent):
        for name, child in list(parent.named_children()):
            if isinstance(child, nn.MultiheadAttention):
                setattr(parent, name, _AttnStub(attn_log, child.embed_dim,
                                                getattr(child, "batch_first", False)))
            else:
                swap(child)
    swap(m)

    counter = FlopCounterMode(display=False)
    with torch.no_grad(), counter:
        m(*inputs)
    for h in handles:
        h.remove()

    cl = counter.get_total_flops()
    at = sum(2 * ((2 * Lq + 2 * Lk) * d * d + 2 * Lq * Lk * d)
             for Lq, Lk, d in attn_log)
    qkav = sum(2 * (2 * Lq * Lk * d) for Lq, Lk, d in attn_log)
    sc = sum(6 * r["ed"] * r["n"] * r["L"] * r["layers"] * (2 if r["bi"] else 1)
             for r in mamba_log)
    tokens = mamba_log[0]["L"] if mamba_log else 0
    del m
    return cl, at, qkav, sc, tokens, len(attn_log)


roi = lambda: torch.randn(1, 6, 1, 64, 64, 64)
model = CrossModalVisionMambaModel().cpu()
cl, at, qkav, sc, tok, n_attn = count_flops(model, [roi(), roi()])
p = sum(q.numel() for q in model.parameters() if q.requires_grad)

print("GFLOPs, one forward pass, batch size 1 -- early cross-modal attention\n")
print(f"  params              {p:,}")
print(f"  tokens per modality {tok:,}")
print(f"  attention calls     {n_attn} (cross-modal, {tok:,} x {tok:,})\n")
print(f"  conv + linear       {cl/1e9:.4f}")
print(f"  attention           {at/1e9:.4f}   (of which QK^T and AV: {qkav/1e9:.4f})")
print(f"  Mamba scan          {sc/1e9:.4f}")
print(f"  {'-'*32}")
print(f"  total               {(cl+at+sc)/1e9:.4f} GFLOPs")

GFLOPs, one forward pass, batch size 1 -- early cross-modal attention

  params              98,370
  tokens per modality 3,072
  attention calls     2 (cross-modal, 3,072 x 3,072)

  conv + linear       0.5285
  attention           2.4663   (of which QK^T and AV: 2.4159)
  Mamba scan          0.1510
  --------------------------------
  total               3.1457 GFLOPs
